In [2]:
import numpy as np
import sys
sys.path.append("../../")
sys.path.append("../../Visualization/")
sys.path.append("../../../")

In [3]:
# experiment_file = '../../experiments/parallelized_experiments/output/voronoi_5/2024_01_07_22_56/experiment_result.json'

# stiffness_path = '../../experiments/parallelized_experiments/output/voronoi_5/2024_01_07_22_56/'
# name = 'voronoi_5'

In [4]:
experiment_file = '../../experiments/parallelized_experiments/output/voronoi_5/2024_01_08_16_21//experiment_result.json'

stiffness_path = '../../experiments/parallelized_experiments/output/voronoi_5/2024_01_08_16_21/'
name = 'voronoi_5'

### Overview

In [5]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

import os
import sys; sys.path.append(os.path.join(os.path.abspath(''), '../../experiments/'));

import experiment_helper
import igl
from periodic_simulation_setup import *
import json

import parallelism, multiprocessing, itertools, setproctitle
import os, time, numpy as np

In [6]:
with open(experiment_file, 'r') as fp:
    data = json.load(fp)

In [7]:
df = pd.DataFrame(data['data'])
fig, axes = plt.subplots(nrows = 1, ncols = 3, figsize = (20, 5))
a = (df.hist('Ipu simulation succeed', ax = axes[0]), df.hist('Planar equilibrium', ax = axes[1]), df.hist('Simulation Kappa value', ax = axes[2]))

In [7]:
np.where(df['Negative stiffness'] == 1)

In [8]:
import visualize_stiffness
import importlib
importlib.reload(visualize_stiffness)

In [9]:
valid_tags = np.array(df['name'][df['Planar equilibrium'] == 1])

In [10]:
len(valid_tags)

In [11]:
import MeshFEM, visualization

In [12]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [29]:
def visualize_pattern(param):
    index = valid_tags[param]
    print(index)
    m = MeshFEM.mesh.Mesh('../../experiments/parallelized_experiments/output/voronoi_5/2024_01_08_16_21//{}/mesh_voronoi_5_{}.obj'.format(index, index))
    fusing_vtx = np.load('../../experiments/parallelized_experiments/output/voronoi_5/2024_01_08_16_21/{}/fusedVtx_voronoi_5_{}.npy'.format(index, index))
    
    visualization.plot_2d_mesh(m, pointList=fusing_vtx, width=5, height=5)

In [30]:
interact(visualize_pattern, param=widgets.IntSlider(min=0, max=len(valid_tags) - 1, step=1, value=0));

In [15]:
kappa_path = None

In [16]:
importlib.reload(visualize_stiffness)

In [17]:
bending_stiffness_data, stretching_stiffness_data, scale_factor_data, used_tags = visualize_stiffness.plot_all_data(kappa_path, stiffness_path, name, valid_tags, plot_data = True, sort_data=False)

In [18]:
parameters = (np.array(data['pattern_parameters'][0]['values']))

In [19]:
max_bending_stiffness = np.max(bending_stiffness_data, axis = 1)
min_bending_stiffness = np.min(bending_stiffness_data, axis = 1)
max_stretching_stiffness = np.max(stretching_stiffness_data, axis = 1)
min_stretching_stiffness = np.min(stretching_stiffness_data, axis = 1)

In [20]:
parameters[np.argmax(min_bending_stiffness)]

In [21]:
x_scale_factors, y_scale_factors = visualize_stiffness.get_axis_scale_factors(stiffness_path, name, valid_tags)

In [22]:
len(x_scale_factors)

In [23]:
# min_scale_factors = np.min(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)
# max_scale_factors = np.max(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)

min_scale_factors = x_scale_factors
max_scale_factors = y_scale_factors

In [24]:
angle_offsets = visualize_stiffness.get_max_flattening_factor_offset(stiffness_path, name, valid_tags)

In [25]:
parameters[(np.where(min_bending_stiffness > 0.1))]

In [26]:
parameters[(np.where(min_stretching_stiffness > 4))]

In [28]:
valid_tags[np.argmax(x_scale_factors)]

### Get scale function convex hull

In [335]:
import matplotlib.cm as cm
import matplotlib as mpl

In [336]:
from scipy.spatial import ConvexHull, convex_hull_plot_2d
import numpy as np
rng = np.random.default_rng()
points = rng.random((30, 2))   # 30 random points in 2-D
# points = np.concatenate((min_scale_factor.reshape((-1, 1)), max_scale_factor.reshape((-1, 1))), axis = 1)

points = np.concatenate((max_scale_factors.reshape((-1, 1)), min_scale_factors.reshape((-1, 1))), axis = 1)
hull = ConvexHull(points)

In [337]:
hull

In [338]:
import matplotlib.pyplot as plt
plt.plot(points[:,0], points[:,1], 'o')
for simplex in hull.simplices:
    plt.plot(points[simplex, 0], points[simplex, 1], 'k-')
plt.plot(points[hull.vertices,0], points[hull.vertices,1], 'r--', lw=2)
plt.plot(points[hull.vertices[0],0], points[hull.vertices[0],1], 'ro')
plt.show()

In [339]:
import numpy.linalg as la

# Need to plot the patches over the min and max scale factors, so we can get the polygon that constrain the singular values
# The scale factors we are considering during the parametrization are from the flattening, so it's the change from the inflated state to the fabricated state, hence we need to take one over the factors we have from the average deformation gradient from homogenization.
# max_scale_factor = 1 / np.array(scale_factor_data)[:, 0]
# min_scale_factor = 1 / np.array(scale_factor_data)[:, 1]

fig, ax = plt.subplots(figsize = (10, 10))

# plt.scatter(x_scale_factor, y_scale_factor, label = data_info[i][1], s = 50, alpha = 0.3)
# plt.scatter(y_scale_factor, x_scale_factor, label = data_info[i][1], s = 50, alpha = 0.3)

# plt.scatter(y_scale_factor, x_scale_factor, label = data_info[i][1], s = 50, alpha = 0.8, c = min_stiffness)


points = np.concatenate((max_scale_factors.reshape((-1, 1)), min_scale_factors.reshape((-1, 1))), axis = 1)
hull = ConvexHull(points)

for simplex in hull.simplices:
    plt.plot(points[simplex, 0], points[simplex, 1], 'k-')

ax.title.set_text("Scale factors")
plt.xlabel("x scale factors")
plt.ylabel("y scale factors")

plt.scatter(max_scale_factors, min_scale_factors, label = 'min_stiffness', s = 200, alpha = 1, c = min_bending_stiffness)
# plt.scatter(x_scale_factors, y_scale_factors, label = 'max_stiffness', s = 200, alpha = 1, c = max_bending_stiffness)

# Plot x = y line
lims = [
np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
]

# now plot both limits against eachother
ax.plot(lims, lims, 'k-', alpha=0.75, zorder=0)
ax.set_aspect('equal')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.legend()
fig.tight_layout()
plt.savefig('scale_factor_values_{}.png'.format(name), dpi = 300)

In [340]:
selector = np.argsort(y_scale_factors)[-1]

In [341]:
index = valid_tags[selector]

In [305]:
max_bending_stiffness[selector], min_bending_stiffness[selector], max_stretching_stiffness[selector], min_stretching_stiffness[selector]

In [306]:
np.array(np.argsort(np.abs(df['Simulation Kappa value'])))[-2]

In [302]:
index = df['name'][np.array(np.argsort(np.abs(df['Simulation Kappa value'])))[-5]]

In [212]:
np.array(np.sort(np.abs(df['Simulation Kappa value'])))[-10:]

In [224]:
np.where(df['Negative stiffness'] == 1)

In [258]:
valid_tags[np.argmax(min_bending_stiffness)]

In [263]:
min_bending_stiffness[np.argmax(min_bending_stiffness)]

In [265]:
y_scale_factors[18], x_scale_factors[18]

In [31]:
index = 17

In [32]:
import inflation

In [11]:
vertices = np.load('../../experiments/parallelized_experiments/output/voronoi_5/2024_01_08_16_21/17/vertices_voronoi_5_17.npy')
edges = np.load('../../experiments/parallelized_experiments/output/voronoi_5/2024_01_08_16_21/17/edges_voronoi_5_17.npy')

In [12]:
visualization.plot_line_segments(vertices, edges - 1)

In [24]:
boundary_vertices = np.array([[-2.5, -2.5, 0], [2.5, -2.5, 0], [2.5, 2.5, 0], [-2.5, 2.5, 0]])

In [25]:
boundary_edges = np.array([[0, 1], [1, 2], [2, 3], [3, 0]]) + len(vertices)

In [28]:
visualization.plot_line_segments(list(vertices[:, :3]) + list(boundary_vertices), list(edges - 1) + list(boundary_edges), width = 3, height = 3)

In [30]:
index = 17

In [32]:
variable = index

m = MeshFEM.mesh.Mesh('../../experiments/parallelized_experiments/output/voronoi_5/2024_01_08_16_21//{}/mesh_voronoi_5_{}.obj'.format(index, index))
fusing_vtx = np.load('../../experiments/parallelized_experiments/output/voronoi_5/2024_01_08_16_21/{}/fusedVtx_voronoi_5_{}.npy'.format(index, index))

vertices = m.vertices()
vertices = np.concatenate((vertices, np.zeros((len(vertices), 1))), axis = 1)
m = MeshFEM.mesh.Mesh(vertices, m.elements())


visualization.plot_2d_mesh(m, pointList=fusing_vtx, width=5, height=5)

ipu = inflation.InflatablePeriodicUnit(m, fusing_vtx, epsilon = 1e-9)
ipu.setVars(np.load('../../experiments/parallelized_experiments/output/voronoi_5/2024_01_08_16_21/{}/voronoi_5_dofs_before_stiffness_{}.npy'.format(index, index)))

viewer = TriMeshViewer(ipu, width=600, height=600)
viewer.showWireframe(False)
viewer.show()

In [261]:
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

optimizer = inflation.get_inflation_optimizer(ipu, ipu.getBendingStiffnessFixedVars(), opts, callback=cb, hessianShift = 0)

#     allow_bending = True
#     curr_vars = ipu.getVars()
#     hessianShiftForAlphainPlanar = 1e-12
#     curr_vars[-1] = np.pi / 2
#     # Push the surface slightly out of plane to escape from the negative stiffness direction.
#     curr_vars[-2] = 0.1

#     bending_fixed_vars = [] if allow_bending else [ipu.numVars() - 2]
#     fixedVars, hessianShift = list(periodic_unit_helper.get_center_fixedVars(ipu)) + bending_fixed_vars, hessianShiftForAlphainPlanar
#     cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)

result_folder = 'out'
if not os.path.exists(result_folder):
    os.makedirs(result_folder)  
render_images = True

ipu_stiffness_values, sampled_alphas, stiffness_coefficient = visualize_sampled_bending_stiffness(ipu, 1000, optimizer, hessianShift = 0, fixedVars = ipu.getBendingStiffnessFixedVars(), filename = "{}/stiffness_{}_{}.png".format(result_folder, name, variable), generate_images = render_images)

In [243]:
from IPython.display import Image
Image(filename="{}/stiffness_{}_{}.png".format(result_folder, name, variable)) 

### Validate the max and min scale factors are aligned with the x and y axis

In [50]:
import visualize_stiffness
importlib.reload(visualize_stiffness)

In [51]:
eqns = hull.equations

In [52]:
hull.max_bound, hull.min_bound

In [55]:
import parametrization_helper, importlib
importlib.reload(parametrization_helper)

In [56]:
parametrization_helper.visualize_scale_factors(eqns, max_scale_factors, min_scale_factors)

### Generate data without augmenting

In [ ]:
import visualize_stiffness
importlib.reload(visualize_stiffness)

In [ ]:
stiffness_coefficients = np.array(visualize_stiffness.get_stiffness_coefficients(stiffness_path, name, (used_tags)))
# For patches with reflection symmetry:
stiffness_coefficients[:, 1] *= 0
stiffness_coefficients[:, 2] *= 0

In [ ]:
# for i in range(5):
#     for j in range(30):
#         stiffness_coefficients[:, i] = parametrization_helper.savitzky_golay(stiffness_coefficients[:, i], 11, 3) # window size 51, polynomial order 3

In [ ]:
plt.plot(stiffness_coefficients[:, 4])

In [ ]:
np.set_printoptions(suppress=True, precision=4)

In [ ]:
np.argmax(stiffness_coefficients[:, 1]), np.argmax(stiffness_coefficients[:, 2])

In [ ]:
def get_stiffness_polynomial(s, theta):
    return s[0] * np.cos(theta)**2 * np.sin(theta)**2 + s[1] * np.cos(theta)**3 * np.sin(theta) + s[2] * np.cos(theta) * np.sin(theta)**3 + s[3] * np.cos(theta)**4 + s[4] * np.sin(theta)**4

In [ ]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [ ]:
titles = ["cos^2 sin^2", "cos^3 sin", "cos sin^3", "cos^4", "sin^4"]

In [ ]:
def get_basis(index, theta):
    if index == 0:
        return np.cos(theta)**2 * np.sin(theta)**2
    if index == 1:
        return np.cos(theta)**3 * np.sin(theta)
    if index == 2:
        return np.cos(theta) * np.sin(theta)**3
    if index == 3:
        return np.cos(theta)**4
    if index == 4:
        return np.sin(theta)**4

In [ ]:
def plot_basis(index):
    sampled_alpha = np.linspace(0, np.pi, 1000)
    sampled_stiffness = get_basis(index, sampled_alpha)
    r = list(sampled_stiffness) + list(sampled_stiffness)
    theta = list(sampled_alpha) + list(np.pi + np.array(sampled_alpha))
    
    
    plot_max_r = None
    plot_min_r = None
    
    fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
    ax.plot(theta, r)
    ax.set_rmax(max(sampled_stiffness) if plot_max_r is None else plot_max_r)
    ax.set_rmin(min(sampled_stiffness) - 0.2 * (max(sampled_stiffness) - min(sampled_stiffness)) if plot_min_r is None else plot_min_r)
    # ax.set_rticks([0.5, 1, 1.5, 2])  # Less radial ticks
    # ax.set_rlabel_position(-22.5)  # Move radial labels away from plotted line
    ax.grid(True)
    
    ax.set_title(titles[index], va='bottom')
    plt.tight_layout()
    plt.savefig("{}.png".format(titles[index]), dpi = 300)

In [ ]:
interact(plot_basis, index=widgets.IntSlider(min=0, max=4, step=1, value=0));

In [ ]:
def plot_stiffness(index):
    sampled_alpha = np.linspace(0, np.pi, 1000)
    coeffs = stiffness_coefficients[index]
    coeffs[1] = 0
    coeffs[2] = 0
    print(coeffs)
    sampled_stiffness = get_stiffness_polynomial(stiffness_coefficients[index], sampled_alpha)
    r = list(sampled_stiffness) + list(sampled_stiffness)
    theta = list(sampled_alpha) + list(np.pi + np.array(sampled_alpha))
    
    
    plot_max_r = None
    plot_min_r = None
    
    fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
    ax.plot(theta, r)
    ax.set_rmax(max(sampled_stiffness) if plot_max_r is None else plot_max_r)
    ax.set_rmin(min(sampled_stiffness) - 0.2 * (max(sampled_stiffness) - min(sampled_stiffness)) if plot_min_r is None else plot_min_r)
    # ax.set_rticks([0.5, 1, 1.5, 2])  # Less radial ticks
    # ax.set_rlabel_position(-22.5)  # Move radial labels away from plotted line
    ax.grid(True)
    
    ax.set_title("Bending stiffness", va='bottom')
    plt.tight_layout()

In [ ]:
interact(plot_stiffness, index=widgets.IntSlider(min=0, max=49, step=1, value=20));

In [ ]:
grid_data = np.zeros((9, len(parameters)))

In [ ]:
for i in range(len(parameters)):
    grid_data[0][i] = max_scale_factors[i]
    grid_data[1][i] = min_scale_factors[i]
    grid_data[2][i] = x_scale_factors[i]
    grid_data[3][i] = y_scale_factors[i]
    for s in range(5):
        grid_data[4 + s][i] = stiffness_coefficients[i][s]

In [ ]:
# np.save("grid_pattern_1.npy", grid_pattern_1)
# np.save("grid_pattern_2.npy", grid_pattern_2)
# np.save("grid_data.npy", grid_data)

In [ ]:
importlib.reload(parametrization_helper)

In [ ]:
splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(grid_data, (parameters))

In [ ]:
grid_data.shape

In [ ]:
scale_factors_grid_data = np.zeros((2, len(parameters)))
for i in range(len(parameters)):
    scale_factors_grid_data[0][i] = x_scale_factors[i]
    scale_factors_grid_data[1][i] = y_scale_factors[i]
scale_factors_splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(scale_factors_grid_data, (parameters))

In [ ]:
test_parameters = np.linspace(0, 0.9, 100)

In [ ]:
fig, axes = plt.subplots(1, 9, figsize=(45, 8))
titles = ['max scale factors', 'min scale factors', 'x scale factors', 'y scale factors', 's1', 's2', 's3', 's4', 's5']
# titles = ['max scale factors', 'min scale factors', 's1', 's2', 's3', 's4', 's5']

for i in range(9):
    axes[i].plot(test_parameters, splines[i * 3 + 1](test_parameters))
    axes[i].set_title(titles[i], fontsize=21)

In [ ]:
stiffness_coefficients = np.array(stiffness_coefficients)

In [ ]:
stiffness_coefficients.shape

In [ ]:
fig, axes = plt.subplots(1, 9, figsize=(45, 8))
titles = ['max scale factors', 'min scale factors', 'x scale factors', 'y scale factors', 's1', 's2', 's3', 's4', 's5']
data = [max_scale_factors, min_scale_factors, x_scale_factors, y_scale_factors, stiffness_coefficients[:, 0], stiffness_coefficients[:, 1], stiffness_coefficients[:, 2], stiffness_coefficients[:, 3], stiffness_coefficients[:, 4]]

for i in range(9):
    axes[i].plot(parameters, data[i])
    axes[i].set_title(titles[i], fontsize=21)

### End data generating

### Parametrization

In [ ]:
import sys; sys.path.append('../../../'); sys.path.append('../../../periodic_patches/'); sys.path.append('../../experiments/'); sys.path.append('../../../gmsh')
import inflation, sparse_matrices, mesh, numpy as np, importlib, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization

In [ ]:
sys.path.append('periodic_patches/')
sys.path.append('gmsh')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
from py_newton_optimizer import NewtonOptimizerOptions

In [ ]:
import MeshFEM, parallelism, benchmark, utils
parallelism.set_max_num_tbb_threads(32)
parallelism.set_gradient_assembly_num_threads(32)
parallelism.set_hessian_assembly_num_threads(32)

In [ ]:
import utils, mesh_utilities
importlib.reload(utils)

In [ ]:
target_surf = mesh.Mesh("../../../../examples/igloo.obj")
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=False))
# target_surf = mesh_utilities.subdivide_loop(target_surf, 1)

In [ ]:
lines = np.array(eqns)

### SIGGRAPH 21 Local global

In [ ]:
# Run some iterations of the local-global algorithm to ensure a good separation between singular values.
# This step can also be used as a prediction of the feasiblity of a design surface:
# if it is unable to nearly satisfy the singular value constraints,
# the surface is probably infeasible.
lg_21 = parametrization.LocalGlobalParametrizer(target_surf, parametrization.lscm(target_surf))

lg_21.alphaMin = 1.4
lg_21.alphaMax = np.pi / 2
print(lg_21.energy())
for i in range(1000): lg_21.runIteration()

print(lg_21.energy())
lg_21.runIteration()
print(lg_21.energy())

In [ ]:
visualization.visualize(lg_21)

In [ ]:
new_lines = np.array([[0, 1, -1.05], [1, 0, -np.pi / 2.], [0, -1, 0.95], [-1., 0, 1.4]])
             # , [0, -1, -1.1], [1, 0, -1.4], [-1, 0, np.pi / 2]]

In [ ]:
parametrization_helper.visualize_scale_factors(new_lines, lg_21.getAlphas(), np.ones(len(lg_21.getAlphas())), )

### New local global to compare

In [ ]:
lg = parametrization.LocalGlobalGenericParametrizer(target_surf, parametrization.lscm(target_surf))

lg.alphaMin = 1.4
lg.alphaMax = np.pi / 2

lg.betaMin = 1.0
lg.betaMax = 1.0

print(lg.energy())
for i in range(1000): lg.runIteration()

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(lg)

### New local global with convex hull

In [ ]:
lg = parametrization.LocalGlobalGenericParametrizer(target_surf, parametrization.lscm(target_surf))

lg.setLines(eqns)

lg.alphaMin = hull.min_bound[0]
lg.alphaMax = hull.max_bound[0]

lg.betaMin = hull.min_bound[1]
lg.betaMax = hull.max_bound[1]

print(lg.energy())
for i in range(1000): lg.runIteration()

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [ ]:
lg.alphaMin, lg.alphaMax, lg.betaMin, lg.betaMax

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(lg, show_main = True)

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, lg.getAlphas(), lg.getBetas())

### Pattern parameters optimization

In [ ]:
default_pattern_params = [0.2]  * len(lg.getAlphas())

In [ ]:
mat_info = np.array(default_pattern_params).reshape((1, len(lg.getAlphas())))

In [ ]:
rparam = parametrization.RegularizedPatternParametrizer(lg, splines, default_pattern_params, len(grid_data.shape) - 1)
rparam.patternParamBounds = np.array([[0.03, 0.4]])
rparam.diffRegW = 0.0

In [ ]:
visualization.visualize_both(rparam, height = 4)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType

In [ ]:
rparam.bendRegW = 1

In [ ]:
rparam.energy(PET.RGP)

In [ ]:
rparam.energy(PET.Bending)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.RGP, PET.Bending]))

In [ ]:
def optimize_rparam(param, patternRegW, phiRegW, bendRegW = 0.0, update_uv = True, niter = 100):
    param.patternRegW = patternRegW
    param.phiRegW = phiRegW
    param.bendRegW = bendRegW
    opts = NewtonOptimizerOptions()
    opts.useIdentityMetric = True
    opts.beta = 1e-4
    opts.niter = niter
    opts.gradTol = 1e-9
    opts.factorizer = opts.factorizer.CatamariNesdis
    benchmark.reset()
    
    if update_uv:
        fixedvars = [param.uOffset(), param.vOffset(), param.phiOffset()]
    else:
        fixedvars = range(param.stretchOffset())

    cr = parametrization.pattern_parametrization_knitro(param, opts.niter, fixedvars)
    benchmark.report()
    return cr

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, 0, 0, bendRegW = 0, update_uv = False, niter = 200)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, rparam.getAlphas(), rparam.getBetas())

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=1)

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 0, phiRegW = 0, bendRegW = 0, update_uv = True, niter = 200)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=1)

In [ ]:
importlib.reload(parametrization_helper)
parametrization_helper.visualize_scale_factors(lines, rparam.getAlphas(), rparam.getBetas())

In [ ]:
# no_regularization_var = rparam.getVars()

In [ ]:
# two_separate_optimization_vars = rparam.getVars()

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 0, phiRegW = 1e-5, bendRegW = 0, update_uv = True, niter = 100)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 0, phiRegW = 1e-5, bendRegW = 0, update_uv = True, niter = 100)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 0, phiRegW = 1e-5, bendRegW = 0, update_uv = True, niter = 100)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=1, width = 30)

In [ ]:
rparam.bendRegW = 1e-3
rparam.patternRegW = 1e-3
rparam.phiRegW = 1e-5
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

### Bending

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-3, phiRegW = 1e-5, bendRegW = 1e-5, update_uv = False, niter = 1000)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-3, phiRegW = 1e-5, bendRegW = 1e-5, update_uv = True, niter = 200)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-3, phiRegW = 1e-5, bendRegW = 1e-5, update_uv = True, niter = 200)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-3, phiRegW = 1e-5, bendRegW = 1e-5, update_uv = True, niter = 200)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=1)

In [ ]:
importlib.reload(parametrization_helper)
parametrization_helper.visualize_scale_factors(lines, rparam.getAlphas(), rparam.getBetas())

In [ ]:
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_VTX, orientationHue=False, width = 5, height = 5)

In [ ]:
# visualization.visualizeChannelOrientationWithIsotropicPoints(rparam, quiver=visualization.QuiverVisualization.PER_TRI, orientationHue=False, width = 5, height = 5, use_x_axis=True)

## Upsampling and channel generation

In [ ]:
import parametrization_helper
importlib.reload(parametrization_helper)

In [ ]:
def fusing_curve_polyline(patternParams):
#     Draw cosine curves.
    amp = patternParams[0]
    def get_y_from_x(x):
        return amp * np.cos(x) * 0.5 * np.pi + np.pi / 2

    x_coords = np.linspace(-np.pi, np.pi, 15)
    y_coords = get_y_from_x(x_coords)
    x_coords += np.pi
    x_coords /= 2
    polyline = np.concatenate(((y_coords).reshape(-1, 1), (x_coords).reshape(-1, 1)), axis = 1)
    return polyline


In [ ]:
sdfVertices, sdfTris, sdf, sheet_vxs, concatenated_polylines, sheet_edges_polylines, boundaryVxs, boundaryEdges = parametrization_helper.get_polyline_from_pattern_parameters(rparam, fusing_curve_polyline, nsubdiv = 7)

In [ ]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, width = 5, height=5)
visualization.plot_line_segments(sheet_vxs, concatenated_polylines, width = 5, height = 5)
visualization.plot_line_segments(list(sheet_vxs) + list(boundaryVxs), list(concatenated_polylines) + list(boundaryEdges + len(sheet_vxs)), width = 5, height = 5)
plt.scatter(boundaryVxs[boundaryEdges[:, 0], 0], boundaryVxs[boundaryEdges[:, 0], 1], c = np.arange(len(boundaryVxs)), cmap = mpl.colormaps['Greys'])

## Meshing and inflation simulation

In [ ]:
import mesher_helper
importlib.reload(mesher_helper)

In [ ]:
import time
time_stamp = time.strftime("%Y_%m_%d_%H_%M")

In [ ]:
np.save("boundary.npy", boundaryVxs[boundaryEdges[:, 0]])

In [ ]:
np.save("sheet_vxs.npy", sheet_vxs)
np.save("concatenated_polylines.npy", concatenated_polylines)

In [ ]:
boundary = boundaryVxs[boundaryEdges[:, 0]][:, :2].tolist()
polylines = []
for polyline in sheet_edges_polylines:
    polyline = np.array(polyline)
    polylines.append(sheet_vxs[np.array(list(polyline[:, 0]) + list([polyline[-1, 1]]))][:, :2].tolist())
parametrization_helper.save_to_svg(boundary, polylines, 'sheet_pattern_{}.svg'.format(time_stamp))

In [ ]:
v, f, fusing_data = mesher_helper.generate_mesh_non_periodic(4, boundaryVxs[boundaryEdges[:, 0]], sheet_vxs, concatenated_polylines, gui = False)

In [ ]:
import numpy as np
import copy

# Use the function
new_v, new_f, new_fusing_without_boundary = parametrization_helper.remove_dangling_vertices(v, f - 1, fusing_data)
m = MeshFEM.mesh.Mesh(new_v, new_f)
new_fusing = copy.copy(new_fusing_without_boundary)
new_fusing[m.boundaryVertices()] = True

In [ ]:
fusing_data, new_fusing

In [ ]:
# m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, SV, SE, triArea=1e0)


In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(new_fusing) == 1)[0], width=10, height=10)


In [ ]:
import inflation
isheet = inflation.InflatableSheet(m, new_fusing)

In [ ]:
from mesh_utilities import SurfaceSampler, tubeRemesh


paramSampler = SurfaceSampler(np.pad(rparam.uv(), [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())

isheet.setUninflatedDeformation(liftedSheetPositions.transpose())

isheet.getVars()

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [ ]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [ ]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [ ]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

fixedVars, hessianShift = [], 1e-6

framerate = 20
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

### First solve with low pressure to get out of indefinite state

In [ ]:
isheet.pressure = 1e-5

In [ ]:
opts.niter = 5

import time
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)

### Then inflate

In [ ]:
isheet.pressure = 5e-2

In [ ]:
opts.niter = 2000
opts.gradTol = 1e-7

import time
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()

In [ ]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(isheet)[:, 0], bins=1000);
plt.xlim(-0.04, 0.1);

In [ ]:
import gzip

In [ ]:
pickle.dump(isheet,  gzip.open("igloo_pattern_optimized_{}_low_frequency_with_bending_high_resolution.pkl.gz".format(time_stamp), 'wb'))

### Generate Fabrication Files

In [ ]:
old_to_new = np.arange(np.max(isheet.wallVertices()) + 1)

In [ ]:
old_to_new[isheet.wallVertices()] = np.arange(len(isheet.wallVertices()))

In [ ]:
from parametrization_helper import form_polylines

In [ ]:
result_vxs = isheet.restWallVertexPositions()
result_edges = old_to_new[isheet.wallBoundaryEdges()]
result_edges = form_polylines(result_edges.tolist())
concatenated_polylines = []
for polyline in result_edges:
    concatenated_polylines.extend(polyline)


In [ ]:
visualization.plot_line_segments(result_vxs, concatenated_polylines)